
# Machine Learning Assignment - 1
## Bike Sharing Demand Prediction

### Student: Pankaj Singh Rawat

---

# Objective

The goal of this assignment is to predict hourly bike rental demand using weather, seasonal, and time-based features.

Models used:
- Linear Regression
- Polynomial Regression (Degree 2)
- Polynomial Regression (Degree 3)
- Ridge Regression
- Lasso Regression

---

# Q1. Examine Dataset Size, Missing Values, and Feature Types

```python
import pandas as pd
import numpy as np

df = pd.read_csv('bike_train.csv')

print(df.shape)
print(df.isnull().sum())
print(df.dtypes)
```

### Observations
- Dataset contains 10,450 rows and 12 columns.
- No missing values are present.
- `datetime` needs conversion to datetime format.

---

# Important Leakage Observation

The columns:
- casual
- registered

directly determine:

count = casual + registered

Therefore, they must be removed before training.

---

# Q2. Visualize Relationships Between Features and Target

```python
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
plt.hist(df['count'], bins=40)
plt.title('Count Distribution')
plt.show()

sns.boxplot(x='season', y='count', data=df)
plt.show()

sns.scatterplot(x='temp', y='count', data=df)
plt.show()
```

### Key Findings
- Temperature positively affects rentals.
- Bad weather reduces rentals.
- Demand varies strongly by time and season.

---

# Q3. Most Informative Variables

Important features:
- hour
- temp
- humidity
- weather
- season
- workingday

These features strongly influence bike demand.

---

# Q4. Feature Engineering

```python
df['datetime'] = pd.to_datetime(df['datetime'])

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['weekday'] = df['datetime'].dt.weekday
df['year'] = df['datetime'].dt.year

df_model = df.drop(columns=['datetime', 'casual', 'registered'])
```

### Why Feature Engineering Helps
These features capture:
- commuting patterns
- seasonal behavior
- weekday/weekend trends

---

# Q5 & Q6. Model Building

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import numpy as np

X = df_model.drop(columns=['count'])
y = df_model['count']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def rmsle(y_true, y_pred):
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),

    'Polynomial Degree 2': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),

    'Polynomial Degree 3': Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),

    'Ridge Regression': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10))
    ]),

    'Lasso Regression': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=0.01, max_iter=10000))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    score = rmsle(y_val, preds)

    results.append([name, score])

results
```

---

# Q7. Model Comparison

| Model | Expected Performance |
|---|---|
| Linear Regression | Baseline |
| Polynomial Degree 2 | Better |
| Polynomial Degree 3 | May overfit |
| Ridge Regression | Stable |
| Lasso Regression | Sparse and regularized |

Lower RMSLE indicates better performance.

---

# Q8. Residual Plot

```python
best_preds = preds

residuals = y_val - best_preds

plt.scatter(best_preds, residuals, alpha=0.5)
plt.axhline(0, color='red')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()
```

### Interpretation
Residuals should be randomly distributed around zero.

---

# Q9. Why the Winning Model Performs Better

The best model performs better because:
- nonlinear relationships are captured
- feature interactions are learned
- regularization reduces overfitting

---

# Q10. Why RMSLE Penalizes Under-Predictions More Gently

RMSLE uses logarithms before computing error.
This reduces the impact of large numeric differences and focuses on relative error.

---

# Q11. Trade-offs Between Simplicity and Predictive Power

### Simple Models
Advantages:
- interpretable
- fast

Disadvantages:
- may underfit

### Complex Models
Advantages:
- higher accuracy

Disadvantages:
- overfitting risk
- less interpretable

---

# Q12. Why Linear Regression Alone Cannot Capture Time-of-Day Effects

Bike demand changes cyclically:
- morning peaks
- evening peaks

Simple linear regression assumes a straight-line relationship and cannot capture these nonlinear patterns effectively.

---

# Final Conclusion

This assignment demonstrates:
- importance of EDA
- importance of feature engineering
- benefits of polynomial regression
- usefulness of regularization
- effectiveness of RMSLE for demand prediction
